## Reflection and Blogpost Writing

## Setup

In [1]:
llm_config = {"model": "gpt-3.5-turbo"}

## The task!

In [2]:
task = '''
        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       '''


## Create a writer agent

In [3]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [4]:
reply = writer.generate_reply(messages=[{"content": task, "role": "user"}])

In [5]:
print(reply)

Title: Unleashing the Power of AI with DeepLearning.AI

Embark on a journey of innovation and discovery with DeepLearning.AI. Founded by AI expert Andrew Ng, this platform offers cutting-edge courses that demystify the world of artificial intelligence and deep learning. Whether you're a novice or seasoned professional, DeepLearning.AI provides the tools and knowledge needed to excel in this fast-evolving field. Join a thriving community of learners, upscale your skills, and unlock a world of possibilities. Dive into the realm of artificial intelligence today with DeepLearning.AI. Let your curiosity meet limitless opportunities!


## Adding reflection 

Create a critic agent to reflect on the work of the writer agent.

In [6]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [7]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

Title: Unleashing the Power of AI with DeepLearning.AI

Embark on a journey of innovation and discovery with DeepLearning.AI. Founded by AI expert Andrew Ng, this platform offers cutting-edge courses that demystify the world of artificial intelligence and deep learning. Whether you're a novice or seasoned professional, DeepLearning.AI provides the tools and knowledge needed to excel in this fast-evolving field. Join a thriving community of learners, upscale your skills, and unlock a world of possibilities. Dive into the realm of artificial intelligence today with DeepLearning.AI. Let your curiosity meet limitless opportunities!

--------------------------------------------------------------------------------
Critic (to Writer):

T

## Nested chat

In [8]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)


In [9]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [10]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [11]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

## Orchestrate the nested chats to solve the task

In [12]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [13]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different task.

In [14]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

Title: Unleashing the Power of AI with DeepLearning.AI

Embark on a journey of innovation and discovery with DeepLearning.AI. Founded by AI expert Andrew Ng, this platform offers cutting-edge courses that demystify the world of artificial intelligence and deep learning. Whether you're a novice or seasoned professional, DeepLearning.AI provides the tools and knowledge needed to excel in this fast-evolving field. Join a thriving community of learners, upscale your skills, and unlock a world of possibilities. Dive into the realm of artificial intelligence today with DeepLearning.AI. Let your curiosity meet limitless opportunities!

--------------------------------------------------------------------------------

*********************

Critic (to Meta Reviewer):

Aggregrate feedback from all reviewers and give final suggestions on the writing.
Context: 
{
  "Reviewer": "SEO Specialist",
  "Review": "Consider adding key terms like 'AI courses,' 'DeepLearning.AI platform,' and 'Andrew Ng AI expert' strategically throughout the content to improve its visibility on search engines. Craft a compelling title and meta description that accurately reflect the content, include target keywords, and entice users to click through when the page appears in search results. Connect to authoritative sources related to AI, deep learning, and Andrew Ng within the content, and also link back to relevant pages on your website to improve SEO performance and user engagement."
}
{
  "Reviewer": "Legal Reviewer",
  "Review": "- Ensure compliance with laws regarding advertising claims and representations, avoiding any misleading statements about the courses or guarantees of success in the field of AI.\n- Verify that there are no unauthorized us

## Get the summary

In [15]:
print(res.summary)

Title: Master AI Skills with DeepLearning.AI by Andrew Ng

Step into the world of artificial intelligence with DeepLearning.AI, a leading platform founded by renowned AI expert Andrew Ng. Explore a myriad of AI courses designed to elevate your expertise in deep learning. Connect with a vibrant community of learners, delve into cutting-edge resources, and unleash your potential in the AI landscape. Discover the transformative power of AI with courses curated by industry experts. Elevate your skills, stay ahead of the curve, and embark on a journey of continuous learning. Join DeepLearning.AI today and shape the future of AI innovation alongside global experts.
